In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv 
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [4]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [ ]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]
    # what does Annotated do and why and when to use it? Annoated is a feature in typing module which allow us to add extra meta data to the type of a entry without affecting the type
    # BaseMessage is a class that represents a message in the chat. It is a base class for all messages. 


In [6]:
def chat_node(state: ChatState) -> ChatState:
  
  messages = state['messages']
  response = model.invoke(messages)
  return {'messages':[response]}

In [7]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id='1'
while True:
    user_input = input("User: ")
    print("User: ",user_input)
    if user_input.strip().lower() in ['exit', 'quit','bye']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)
    print("Bot:", response['messages'][-1].content)

# without persistance the problem was that in while loop the state of the conversation was not being saved, so each time the user input a message, the chatbot would only see that single message and respond to it without any context from previous messages. 


User:  i am Aditya
Bot: Nice to meet you, Aditya! Is there something I can help you with or would you like to chat?
User:  what is my name?
Bot: Your name is Aditya.
User:  exit


In [10]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='i am Aditya', additional_kwargs={}, response_metadata={}, id='fa19b710-31df-44cd-a398-ad6537f2c7a9'), AIMessage(content='Nice to meet you, Aditya! Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 15, 'total_tokens': 40}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0d81d-4099-7d81-b789-848367beae15-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 25, 'total_tokens': 40}), HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id='a2eae8f7-13f9-44d4-8924-1e572ee42023'), AIMessage(content='Your name is Aditya.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 54, 'total_tokens': 62}, 'model_name': 'met